**NAME:**

**ROLL NO.:**

**BATCH:**

**SI-PROJECT**

**House Rent Variation and Property Characteristics Analysis**

In [ ]:
#Install libraries (run once)
!pip install -q kagglehub gradio

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

**Dataset:** House Rent Prediction Dataset (Kaggle) - about 4700 houses/flats for rent in 6 Indian cities.

Columns: Posted On, BHK, Rent, Size, Floor, Area Type, Area Locality, City, Furnishing Status, Tenant Preferred, Bathroom, Point of Contact.

In [ ]:
#Load dataset from Kaggle
import kagglehub
path = kagglehub.dataset_download("iamsouravbanerjee/house-rent-prediction-dataset")
df = pd.read_csv(path + "/House_Rent_Dataset.csv")
print(df.shape)
df.head()

In [ ]:
#If the Kaggle download does not work: download the CSV from Kaggle, upload it
#in the Colab Files panel (left side), then run only these two lines
#df = pd.read_csv("House_Rent_Dataset.csv")
#print(df.shape)

In [ ]:
#PREPROCESSING

In [ ]:
df.columns = df.columns.str.replace(" ", "_")
print(df.columns)

In [ ]:
df.info()

In [ ]:
print("Missing values:")
print(df.isnull().sum())

In [ ]:
#CLEANING

In [ ]:
df = df.dropna()
df = df.drop_duplicates()
df["Posted_On"] = pd.to_datetime(df["Posted_On"])
df = df[(df["Rent"] > 0) & (df["Size"] > 0)]

In [ ]:
print("Clean data shape:", df.shape)
print("Missing values left:", df.isnull().sum().sum())

In [ ]:
#Summary: types of data and scales

In [ ]:
print("Total houses:", len(df))
print("Numerical columns: Rent, Size (continuous, ratio scale), BHK, Bathroom (discrete, ratio scale)")
print("Categorical columns: City, Furnishing_Status, Area_Type, Tenant_Preferred, Point_of_Contact (nominal scale)")
print()
print(df["City"].value_counts())

In [ ]:
#PR2

In [ ]:
#Rent statistics

In [ ]:
r = df["Rent"]
print("Mean =", r.mean())
print("Median =", r.median())
print("Mode =", r.mode()[0])
print("Variance =", r.var())
print("Std Dev =", r.std())
print("Range =", r.max() - r.min())
print("Skewness =", r.skew())

In [ ]:
df[["Rent", "Size", "BHK", "Bathroom"]].describe()

In [ ]:
#City-wise average rent

In [ ]:
print(df.groupby("City")["Rent"].mean().sort_values(ascending=False))

In [ ]:
#Furnishing-wise average rent

In [ ]:
print(df.groupby("Furnishing_Status")["Rent"].mean().sort_values(ascending=False))

In [ ]:
#Interpretation

The mean of Rent is higher than the median and the skewness is positive, so the data is right-skewed (Mean > Median > Mode).
A few very costly houses pull the mean up, while most houses have a lower rent.
The standard deviation is large, so rent varies a lot.

In [ ]:
#Histogram of Rent

In [ ]:
plt.hist(df["Rent"], bins=50, edgecolor="black", range=(0, 150000))
plt.title("Histogram of Rent (up to 1.5 lakh)")
plt.xlabel("Rent")
plt.ylabel("Frequency")
plt.show()

In [ ]:
#Number of houses in each city

In [ ]:
df["City"].value_counts().plot(kind="bar", edgecolor="black")
plt.title("Number of Houses by City")
plt.xlabel("City")
plt.ylabel("Count")
plt.show()

In [ ]:
#Average rent by city

In [ ]:
df.groupby("City")["Rent"].mean().sort_values().plot(kind="bar", edgecolor="black")
plt.title("Average Rent by City")
plt.xlabel("City")
plt.ylabel("Average Rent")
plt.show()

In [ ]:
#Furnishing status share

In [ ]:
fs = df["Furnishing_Status"].value_counts()
plt.pie(fs, labels=fs.index, autopct="%1.1f%%")
plt.title("Furnishing Status Share")
plt.show()

In [ ]:
#Scatter plot

In [ ]:
plt.scatter(df["Size"], df["Rent"])
plt.title("Size vs Rent")
plt.xlabel("Size (sq ft)")
plt.ylabel("Rent")
plt.show()

In [ ]:
#Interpretation

The histogram is right skewed: most houses have a low or medium rent.
Mumbai (and a few other big cities) have the highest average rent.
Most houses are semi-furnished or unfurnished.
Size and Rent generally increase together.

In [ ]:
#Z-score and Empirical Rule

In [ ]:
mu = df["Rent"].mean()
sigma = df["Rent"].std()

df["Rent_Z"] = (df["Rent"] - mu) / sigma

for k in [1, 2, 3]:
    pct = (df["Rent_Z"].abs() <= k).mean() * 100
    print("Within", k, "SD:", round(pct, 2), "%")

print("Normal distribution would give 68.3%, 95.5% and 99.7%")

In [ ]:
#Z-score of one rent value (example: Rs. 50000)

In [ ]:
x = 50000
z = (x - mu) / sigma
print("z-score of 50000 =", z)
print("Houses with rent above 50000 (%):", (df["Rent"] > x).mean() * 100)

In [ ]:
#Outliers (z-score above 3)

In [ ]:
print("Outliers:", (df["Rent_Z"].abs() > 3).sum())

In [ ]:
#Log transformation

In [ ]:
df["Log_Rent"] = np.log(df["Rent"])

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(df["Rent"], bins=40, range=(0, 150000), edgecolor="black")
plt.title("Rent (skewed)")
plt.subplot(1, 2, 2)
plt.hist(df["Log_Rent"], bins=40, edgecolor="black")
plt.title("Log of Rent (bell shaped)")
plt.show()

print("Skewness of Rent =", df["Rent"].skew())
print("Skewness of Log Rent =", df["Log_Rent"].skew())

In [ ]:
#Q-Q plot

In [ ]:
stats.probplot(df["Log_Rent"], dist="norm", plot=plt)
plt.title("Q-Q plot of Log Rent")
plt.show()

In [ ]:
#Interpretation

The real Rent data does not follow the 68-95-99.7 rule of a normal distribution because it is right skewed.
After taking the log, the skewness comes close to 0 and the Q-Q plot is close to a straight line,
so Rent is approximately log-normal.

In [ ]:
#Correlation

In [ ]:
pairs = [("Size", "Rent"), ("BHK", "Rent"), ("Bathroom", "Rent")]

for a, b in pairs:
    x, y = df[a], df[b]
    r, p1 = stats.pearsonr(x, y)
    rho, p2 = stats.spearmanr(x, y)
    print(a, "vs", b)
    print("Covariance =", x.cov(y))
    print("Pearson r =", r, ", p =", p1)
    print("Spearman =", rho, ", p =", p2)
    print("Decision:", "Reject H0 (Significant)" if p1 < 0.05 else "Fail to reject H0")
    print()

In [ ]:
#Correlation heatmap

In [ ]:
sns.heatmap(df[["Rent", "Size", "BHK", "Bathroom"]].corr(), annot=True)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
#Interpretation

Size, BHK and Bathroom all have a positive relationship with Rent: a bigger house with more rooms has a higher rent.
Covariance only shows the direction; Pearson and Spearman show the strength.
If the p-values are below 0.05, the correlations are significant.

In [ ]:
#Sampling and Distributions

In [ ]:
#Sampling methods

In [ ]:
np.random.seed(1)

# Simple random sampling
sample1 = df.sample(n=200, random_state=1)

# Systematic sampling (every 20th row)
sample2 = df.iloc[::20]

# Stratified sampling (30 houses from each City)
sample3 = df.groupby("City").sample(n=30, random_state=1)

# Cluster sampling (2 random Cities)
clusters = np.random.choice(df["City"].unique(), 2, replace=False)
print("Selected clusters:", clusters)
sample4 = df[df["City"].isin(clusters)]

# Convenience sampling (first 200 rows)
sample5 = df.head(200)

In [ ]:
#Compare population and samples

In [ ]:
comparison = pd.DataFrame({
    "Statistics": ["Size", "Mean", "Median", "Std Dev"],
    "Population": [len(df), df["Rent"].mean(), df["Rent"].median(), df["Rent"].std()],
    "Simple Random": [len(sample1), sample1["Rent"].mean(), sample1["Rent"].median(), sample1["Rent"].std()],
    "Systematic": [len(sample2), sample2["Rent"].mean(), sample2["Rent"].median(), sample2["Rent"].std()],
    "Stratified": [len(sample3), sample3["Rent"].mean(), sample3["Rent"].median(), sample3["Rent"].std()],
    "Cluster": [len(sample4), sample4["Rent"].mean(), sample4["Rent"].median(), sample4["Rent"].std()],
    "Convenience": [len(sample5), sample5["Rent"].mean(), sample5["Rent"].median(), sample5["Rent"].std()]
})
comparison

In [ ]:
#Percentage error in mean

In [ ]:
population_mean = df["Rent"].mean()

for name, s in [("Simple Random", sample1), ("Systematic", sample2), ("Stratified", sample3),
                ("Cluster", sample4), ("Convenience", sample5)]:
    error = abs((s["Rent"].mean() - population_mean) / population_mean) * 100
    print(name, "- Percentage Error in Mean:", round(error, 2))

In [ ]:
#Binomial distribution

In [ ]:
p = (df["Furnishing_Status"] == "Furnished").mean()
print("Probability of a furnished house:", p)

binomial = np.random.binomial(n=10, p=p, size=1000)

plt.figure(figsize=(6, 4))
plt.hist(binomial, bins=11, edgecolor="black")
plt.title("Binomial Distribution")
plt.xlabel("Number of Furnished Houses out of 10")
plt.ylabel("Frequency")
plt.show()

print("Mean = np =", 10 * p)
print("Variance = npq =", 10 * p * (1 - p))

In [ ]:
#Poisson distribution

In [ ]:
lambda_value = len(df) / df["Posted_On"].nunique()
print("Lambda =", lambda_value)

poisson = np.random.poisson(lam=lambda_value, size=1000)

plt.figure(figsize=(6, 4))
plt.hist(poisson, bins=15, edgecolor="black")
plt.title("Poisson Distribution")
plt.xlabel("Number of Houses Posted per Day")
plt.ylabel("Frequency")
plt.show()

In [ ]:
#Interpretation

Simple random and stratified sample means are usually close to the population mean.
Convenience sampling (first rows only) can be biased because it does not cover the whole data.
Stratified sampling covers all cities equally.
The binomial chart shows how many furnished houses we get in 10 random houses.
The Poisson chart shows the number of new listings posted in one day.

In [ ]:
#CLT and Point Estimation

In [ ]:
#Population

In [ ]:
population = df["Rent"].values
population_mean = np.mean(population)
population_std = np.std(population)

print("Population Mean:", population_mean)
print("Population Std:", population_std)

plt.figure(figsize=(8, 5))
plt.hist(population, bins=50, range=(0, 150000))
plt.title("Population Distribution (Right-Skewed)")
plt.xlabel("Rent")
plt.ylabel("Frequency")
plt.show()

In [ ]:
#Repeated sampling, n = 30

In [ ]:
np.random.seed(50)
sample_size = 30
number_of_sample = 1000
sample_means = []

for i in range(number_of_sample):
    sample = np.random.choice(population, size=sample_size)
    sample_means.append(np.mean(sample))

sample_means = np.array(sample_means)

In [ ]:
#Distribution of sample means

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(sample_means, bins=30, edgecolor="black")
plt.axvline(population_mean, linestyle="--", color="red", label="Population Mean")
plt.title("Distribution of Sample Means (n=30)")
plt.xlabel("Sample Mean")
plt.ylabel("Frequency")
plt.legend()
plt.show()

In [ ]:
#CLT results and standard error

In [ ]:
observed_se = np.std(sample_means, ddof=1)
theoretical_se = population_std / np.sqrt(sample_size)

print("CLT Results")
print("Population Mean:", population_mean)
print("Mean of Sample Means:", np.mean(sample_means))
print("Observed Standard Error:", observed_se)
print("Theoretical Standard Error:", theoretical_se)

In [ ]:
#Effect of sample size

In [ ]:
sizes = [5, 30, 100]
plt.figure(figsize=(10, 5))

for n in sizes:
    means = [np.mean(np.random.choice(population, size=n)) for i in range(1000)]
    plt.hist(means, bins=30, alpha=0.5, label=f"n={n}")

plt.title("Sample Means for Different Sample Sizes")
plt.xlabel("Sample Mean")
plt.ylabel("Frequency")
plt.legend()
plt.show()

In [ ]:
#Point estimation

In [ ]:
sample = df["Rent"].sample(n=50, random_state=42)

estimated_mean = sample.mean()
estimated_variance = sample.var()
estimated_std = sample.std()

print("Point Estimation")
print("Estimated Mean:", estimated_mean)
print("Estimated Variance:", estimated_variance)
print("Estimated Standard Deviation:", estimated_std)

error = abs(estimated_mean - population_mean)
print("Estimation Error:", error)

In [ ]:
#Interpretation

The population is right skewed, but the sample means look normal. This proves the Central Limit Theorem.
The mean of sample means is close to the population mean.
Observed SE is close to theoretical SE (sigma / sqrt(n)).
A bigger sample size gives a narrower spread, so a more accurate estimate.

In [ ]:
#Confidence Interval

In [ ]:
#Using a sample of 100 houses

In [ ]:
sample = df["Rent"].sample(n=100, random_state=1)
n = len(sample)
sample_mean = sample.mean()
sample_std = sample.std()
confidence = 0.95

# Z interval (population standard deviation known)
z_critical = stats.norm.ppf((1 + confidence) / 2)
margin_z = z_critical * population_std / np.sqrt(n)
print("Z interval:", sample_mean - margin_z, "to", sample_mean + margin_z)
print("Margin of error (Z):", margin_z)

# T interval (sample standard deviation used)
t_critical = stats.t.ppf((1 + confidence) / 2, df=n - 1)
margin_t = t_critical * sample_std / np.sqrt(n)
print("T interval:", sample_mean - margin_t, "to", sample_mean + margin_t)
print("Margin of error (T):", margin_t)

print("True population mean:", population_mean)

We are 95% confident that the true mean rent lies between the lower and upper limit.
A higher confidence level (99%) or a smaller sample gives a wider interval.

In [ ]:
#Hypothesis Tests

In [ ]:
#Z-test (population std known, n = 100 > 30)
#H0: mean rent = 30000    H1: mean rent is not 30000

In [ ]:
sample = df["Rent"].sample(n=100, random_state=1)
mu0 = 30000
sigma = df["Rent"].std(ddof=0)

z = (sample.mean() - mu0) / (sigma / np.sqrt(len(sample)))
p = 2 * (1 - stats.norm.cdf(abs(z)))

print("Sample mean =", sample.mean())
print("z-statistic =", z)
print("Critical values = -1.96 and +1.96")
print("p-value =", p)
print("Decision:", "Reject H0" if p < 0.05 else "Fail to reject H0")

In [ ]:
#T-test (population std unknown, small sample n = 25)
#H0: mean rent = 30000    H1: mean rent is not 30000

In [ ]:
small_sample = df["Rent"].sample(n=25, random_state=3)

t, p = stats.ttest_1samp(small_sample, 30000)
print("Sample mean =", small_sample.mean())
print("Degrees of freedom =", len(small_sample) - 1)
print("t-statistic =", t)
print("p-value =", p)
print("Decision:", "Reject H0" if p < 0.05 else "Fail to reject H0")

In [ ]:
#Two-sample t-test: Furnished vs Unfurnished houses

In [ ]:
furnished = df[df["Furnishing_Status"] == "Furnished"]["Rent"]
unfurnished = df[df["Furnishing_Status"] == "Unfurnished"]["Rent"]

t, p = stats.ttest_ind(furnished, unfurnished)
print("Furnished mean =", furnished.mean())
print("Unfurnished mean =", unfurnished.mean())
print("t-statistic =", t)
print("p-value =", p)
print("Decision:", "Reject H0" if p < 0.05 else "Fail to reject H0")

In [ ]:
#Interpretation

Rule used: Z-test when sigma is known and n > 30, otherwise T-test.
If p < 0.05 we reject H0, which means there is enough evidence that the mean is different from the claimed value.
For the two-sample test, if p < 0.05 then furnished and unfurnished houses have significantly different average rent.

In [ ]:
#Chi-Square Goodness of Fit
#H0: Furnishing status follows the expected distribution (20% Furnished, 40% Semi-Furnished, 40% Unfurnished)

In [ ]:
observed = df["Furnishing_Status"].value_counts().sort_index()
print(observed)

expected_share = np.array([0.2, 0.4, 0.4])
expected = expected_share * observed.sum()

chi2, p = stats.chisquare(f_obs=observed, f_exp=expected)
print("Expected counts:", expected)
print("Chi-Square:", chi2)
print("Degrees of freedom:", len(observed) - 1)
print("p-value:", p)
print("Decision:", "Reject H0" if p < 0.05 else "Fail to reject H0")

If p < 0.05, the real furnishing mix is different from the expected mix.

In [ ]:
#ANOVA

In [ ]:
#Rent across Furnishing Status
#H0: all means are equal    H1: at least one mean is different

In [ ]:
groups = [grp["Rent"].values for name, grp in df.groupby("Furnishing_Status")]
F, p = stats.f_oneway(*groups)

print("F-statistic =", F)
print("p-value =", p)
print("Decision:", "Reject H0" if p < 0.05 else "Fail to reject H0")

In [ ]:
#Rent across Cities

In [ ]:
groups = [grp["Rent"].values for name, grp in df.groupby("City")]
F, p = stats.f_oneway(*groups)

print("F-statistic =", F)
print("p-value =", p)
print("Decision:", "Reject H0" if p < 0.05 else "Fail to reject H0")

If p < 0.05, average rent differs across the groups (furnishing status / cities).
Otherwise, that factor has no significant effect on rent.

## Classification: Rent Category Prediction

**Problem:** classify a house as **Low**, **Medium** or **High** rent using its characteristics (size, BHK, bathrooms, floor, city, furnishing status, area type, tenant preferred and point of contact).

**Target variable:** `Rent_Category`, made by splitting Rent into three equal-sized groups (tertiles). Rent itself is **not** used as an input, so the models cannot see the answer.

**Models:** Logistic Regression, Decision Tree and Random Forest. They are compared with accuracy, precision, recall, F1-score and confusion matrix. The best model is saved and used in the dashboard.

In [ ]:
#CLASSIFICATION: imports
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
import joblib
import sklearn
print("scikit-learn version:", sklearn.__version__)

In [ ]:
#Create the target variable (Low / Medium / High rent)
clf_df = df.copy()
clf_df["Rent_Category"], rent_edges = pd.qcut(clf_df["Rent"], 3, labels=["Low", "Medium", "High"], retbins=True)

print("Rent limits (Rs.):", rent_edges)
print()
print("Class distribution:")
print(clf_df["Rent_Category"].value_counts())

clf_df["Rent_Category"].value_counts().reindex(["Low", "Medium", "High"]).plot(kind="bar", edgecolor="black", color=["#4ade80", "#facc15", "#f87171"])
plt.title("Class Distribution of Rent Category")
plt.xlabel("Rent Category")
plt.ylabel("Count")
plt.show()

**Preprocessing for classification**
1. `Floor` is text (for example "3 out of 5"), so it is split into two numbers: `Floor_No` and `Total_Floors`.
2. Categorical columns are converted to numbers with one-hot encoding (`get_dummies`).
3. `Area_Locality` (too many different values) and `Posted_On` are not used.
4. Data is split 80% train / 20% test with stratification, so every class has the same share in both parts.
5. Feature scaling (`StandardScaler`) is applied inside a Pipeline for Logistic Regression, which is affected by feature scale. Tree models do not need it.

In [ ]:
#Preprocessing: Floor column, encoding, train-test split
def floor_no(s):
    first = str(s).split(" out of ")[0].strip()
    if first == "Ground":
        return 0
    if "Basement" in first:
        return -1
    try:
        return int(first)
    except ValueError:
        return 0


def total_floors(s):
    try:
        return int(str(s).split(" out of ")[1])
    except (IndexError, ValueError):
        return 1


clf_df["Floor_No"] = clf_df["Floor"].apply(floor_no)
clf_df["Total_Floors"] = clf_df["Floor"].apply(total_floors)

num_cols = ["BHK", "Size", "Bathroom", "Floor_No", "Total_Floors"]
cat_cols = ["City", "Furnishing_Status", "Area_Type", "Tenant_Preferred", "Point_of_Contact"]

X = pd.get_dummies(clf_df[num_cols + cat_cols], columns=cat_cols, drop_first=True).astype(float)
y = clf_df["Rent_Category"].astype(str)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Features used:", X.shape[1])
print("Train size:", X_train.shape, " Test size:", X_test.shape)
X.head()

**Feature selection:** all features above are kept because the dataset is small and has few columns. Random Forest feature importance (shown later) tells us which features matter most.

In [ ]:
#Train multiple classification models
models = {
    "Logistic Regression": Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1000))]),
    "Decision Tree": Pipeline([("model", DecisionTreeClassifier(max_depth=8, random_state=42))]),
    "Random Forest": Pipeline([("model", RandomForestClassifier(n_estimators=200, random_state=42))]),
}

labels = ["Low", "Medium", "High"]
rows = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred
    cv = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy").mean()
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, pred, average="weighted", zero_division=0),
        "F1-Score": f1_score(y_test, pred, average="weighted", zero_division=0),
        "CV Accuracy (5-fold)": cv,
    })

results = pd.DataFrame(rows).set_index("Model").sort_values("F1-Score", ascending=False)
results.round(4)

In [ ]:
#Model comparison chart
results[["Accuracy", "Precision", "Recall", "F1-Score"]].plot(kind="bar", figsize=(10, 5), edgecolor="black")
plt.title("Model Comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.legend(loc="lower right")
plt.show()

In [ ]:
#Confusion matrix of every model
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes = axes.ravel()

for i, name in enumerate(models):
    cm = confusion_matrix(y_test, predictions[name], labels=labels)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=axes[i], cbar=False)
    axes[i].set_title(name)
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("Actual")

plt.tight_layout()
plt.show()

In [ ]:
#Best model selection (highest weighted F1-score)
best_name = results.index[0]
best_model = models[best_name]

print("Best model:", best_name)
print("Accuracy :", round(results.loc[best_name, "Accuracy"], 4))
print("F1-Score :", round(results.loc[best_name, "F1-Score"], 4))
print()
print(classification_report(y_test, predictions[best_name], labels=labels, zero_division=0))

cm = confusion_matrix(y_test, predictions[best_name], labels=labels)
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=labels, yticklabels=labels)
plt.title("Confusion Matrix - " + best_name)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
#Feature importance (Random Forest)
rf = models["Random Forest"].named_steps["model"]
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
imp.sort_values().plot(kind="barh", edgecolor="black", color="#6366f1")
plt.title("Top 10 Important Features (Random Forest)")
plt.xlabel("Importance")
plt.show()

In [ ]:
#Save the best model and test loading it
joblib.dump({"model": best_model, "columns": X.columns.tolist(), "rent_edges": rent_edges.tolist(),
             "model_name": best_name}, "best_rent_model.pkl")

saved = joblib.load("best_rent_model.pkl")
sample_row = X_test.iloc[[0]]
print("Saved model:", saved["model_name"])
print("Test prediction:", saved["model"].predict(sample_row)[0], "| Actual:", y_test.iloc[0])

#To keep the model file in Colab, download it (optional)
try:
    from google.colab import files
    files.download("best_rent_model.pkl")
except Exception:
    pass

**Interpretation (classification)**
- Rent category is predicted from house characteristics only; Rent is not used as an input.
- Models are compared on accuracy, precision, recall and F1-score (weighted over the three classes), and the model with the highest F1-score is selected as best.
- Most mistakes in the confusion matrix are between neighbouring classes (Low-Medium and Medium-High), which is expected because the classes are cut from one continuous rent scale.
- The best model is saved as `best_rent_model.pkl` and used in the dashboard's Rent Category Predictor tab.

## Regression: Predicting the Exact Monthly Rent

**Problem:** predict the exact monthly rent (Rs.) of a house from its characteristics. This is a regression problem, because the output is a continuous number and not a category.

**Target variable:** `Rent` itself. The same input features as the classification part are used, and Rent is not one of them.

**Models:** Linear Regression and Random Forest Regressor.

**Metrics:**
- **R²**: share of the variation in rent that the model explains (closer to 1 is better).
- **MAE**: average error in rupees (closer to 0 is better).
- **RMSE**: like MAE but penalises large errors more (closer to 0 is better).

In [ ]:
#REGRESSION: imports, split and training
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

y_reg = clf_df["Rent"]
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X, y_reg, test_size=0.2, random_state=42)
print("Train size:", Xr_train.shape, " Test size:", Xr_test.shape)

reg_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=200, random_state=42),
}

reg_rows = []
reg_preds = {}

for name, model in reg_models.items():
    model.fit(Xr_train, yr_train)
    pred = model.predict(Xr_test)
    reg_preds[name] = pred
    cv_r2 = cross_val_score(model, Xr_train, yr_train, cv=5, scoring="r2").mean()
    reg_rows.append({
        "Model": name,
        "R2": r2_score(yr_test, pred),
        "MAE (Rs.)": mean_absolute_error(yr_test, pred),
        "RMSE (Rs.)": np.sqrt(mean_squared_error(yr_test, pred)),
        "CV R2 (5-fold)": cv_r2,
    })

reg_results = pd.DataFrame(reg_rows).set_index("Model").sort_values("R2", ascending=False)
reg_results.round(3)

In [ ]:
#Actual vs predicted rent
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, name in zip(axes, reg_models):
    pred = reg_preds[name]
    top = max(yr_test.quantile(0.99), np.quantile(pred, 0.99))
    ax.scatter(yr_test, pred, s=10, alpha=0.4, color="#4338ca")
    ax.plot([0, top], [0, top], "r--", label="Perfect prediction")
    ax.set_xlim(0, top)
    ax.set_ylim(0, top)
    ax.set_title(name)
    ax.set_xlabel("Actual Rent (Rs.)")
    ax.set_ylabel("Predicted Rent (Rs.)")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
#Best regression model and feature importance (Random Forest Regressor)
best_reg_name = reg_results.index[0]
print("Best regression model:", best_reg_name)
print(reg_results.loc[[best_reg_name]].round(3))

rfr = reg_models["Random Forest Regressor"]
imp_r = pd.Series(rfr.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
imp_r.sort_values().plot(kind="barh", edgecolor="black", color="#16a34a")
plt.title("Top 10 Important Features for Rent (Random Forest Regressor)")
plt.xlabel("Importance")
plt.show()

**Interpretation (regression)**
- R² shows how much of the rent variation the model explains; MAE and RMSE show the typical error in rupees.
- Linear Regression assumes a straight-line relationship between features and rent. Random Forest Regressor can capture non-linear patterns, so it is expected to fit better.
- Rent is right-skewed, so errors are usually larger for expensive houses. The actual vs predicted chart shows this as points spreading out at high rent.
- The best regression model is the one with the highest R² on the test set.

## Interactive Dashboard (Gradio, public link)
Three tabs:
1. **Rent Checker** - is my rent cheap, fair or costly compared with similar houses?
2. **Data Explorer** - filter the data, see summary cards and charts, download the filtered CSV
3. **Rent Category Predictor** - predicts Low / Medium / High rent with the best classification model

In [ ]:
#Dashboard part 1: helper functions and calculations
import gradio as gr

CITIES = sorted(df["City"].unique())
FURN = sorted(df["Furnishing_Status"].unique())
BHKS = sorted(int(b) for b in df["BHK"].unique())
RENT_CAP = int(df["Rent"].quantile(0.99))


# ---------- small HTML helpers (every text has its own colour, so it is readable in light and dark mode) ----------
def inr(x):
    return "₹" + format(round(x), ",")


def card(title, big, sub, color):
    dark = "color:#111827 !important;"
    return ('<div style="background:' + color + ';border-radius:14px;padding:16px 20px;font-family:sans-serif;' + dark + '">'
            '<div style="font-size:12px;letter-spacing:1px;text-transform:uppercase;font-weight:600;' + dark + '">' + title + '</div>'
            '<div style="font-size:28px;font-weight:700;margin:2px 0;' + dark + '">' + big + '</div>'
            '<div style="font-size:14px;' + dark + '">' + sub + '</div></div>')


def tiles(items):
    html = ""
    for label, value in items:
        html += ('<div style="flex:1;min-width:130px;background:#312e81;border-radius:12px;padding:10px 12px;'
                 'text-align:center;font-family:sans-serif">'
                 '<div style="font-size:12px;color:#e0e7ff !important;">' + str(label) + '</div>'
                 '<div style="font-size:19px;font-weight:700;color:#ffffff !important;">' + str(value) + '</div></div>')
    return '<div style="display:flex;gap:10px;flex-wrap:wrap">' + html + '</div>'


def bars(items):
    html = '<div style="background:#1f2937;border-radius:12px;padding:10px 16px;font-family:sans-serif">'
    for label, pct, color in items:
        html += ('<div style="margin:8px 0;font-size:13px"><div style="display:flex;justify-content:space-between">'
                 '<span style="color:#f9fafb !important;">' + label + '</span>'
                 '<b style="color:#ffffff !important;">' + format(pct, ".0f") + '%</b></div>'
                 '<div style="background:#4b5563;border-radius:8px;height:10px">'
                 '<div style="width:' + format(min(pct, 100), ".0f") + '%;background:' + color + ';height:10px;border-radius:8px"></div></div></div>')
    return html + '</div>'


def done(fig):
    plt.close(fig)
    return fig


# ---------- Tab 1: Rent Checker ----------
def rent_check(city, furn, bhk, size, rent, conf):
    bhk = int(bhk)
    d = df[(df["City"] == city) & (df["Furnishing_Status"] == furn) & (df["BHK"] == bhk)]
    note = "Compared with " + str(len(d)) + " " + furn.lower() + " " + str(bhk) + " BHK houses in " + city + "."
    if len(d) < 10:
        d = df[(df["City"] == city) & (df["BHK"] == bhk)]
        note = "Few exact matches, so all " + str(bhk) + " BHK houses in " + city + " were used (" + str(len(d)) + " houses)."
    if len(d) < 10:
        d = df[df["City"] == city]
        note = "Few matches, so all " + str(len(d)) + " houses in " + city + " were used."

    r = d["Rent"]
    n, mean, sd = len(r), r.mean(), r.std()
    pct = (r < rent).mean() * 100                          # percentile of your rent
    z = (rent - mean) / sd                                 # z-score
    me = stats.t.ppf((1 + conf / 100) / 2, df=n - 1) * sd / np.sqrt(n)   # margin of error
    ppsf = (d["Rent"] / d["Size"])
    est = ppsf.median() * size                             # typical rent for your size

    if pct < 25:
        verdict, color = "Cheap - good deal", "#4ade80"
    elif pct <= 75:
        verdict, color = "Fair price", "#a3e635"
    elif pct <= 90:
        verdict, color = "Costly", "#fb923c"
    else:
        verdict, color = "Very costly", "#f87171"
    c1 = card("Price verdict", verdict, "Your rent " + inr(rent) + " is higher than " + format(pct, ".0f") + "% of similar houses.<br>" + note, color)

    k = tiles([("Average rent (similar)", inr(mean)), ("Median rent", inr(r.median())),
               ("Typical rent for " + str(int(size)) + " sq ft", inr(est)), ("Your z-score", format(z, ".2f")),
               (str(int(conf)) + "% CI for mean", inr(mean - me) + " - " + inr(mean + me)),
               ("Your rent / sq ft", inr(rent / size))])

    b = bars([("Similar houses cheaper than yours (>10% less)", (r < rent * 0.9).mean() * 100, "#4ade80"),
              ("Similar houses at about your price (±10%)", ((r >= rent * 0.9) & (r <= rent * 1.1)).mean() * 100, "#facc15"),
              ("Similar houses costlier than yours (>10% more)", (r > rent * 1.1).mean() * 100, "#f87171")])

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].hist(r[r <= r.quantile(0.99)], bins=25, color="#6366f1", edgecolor="white")
    ax[0].axvspan(mean - me, mean + me, color="#fde047", alpha=0.6, label="CI of mean")
    ax[0].axvline(r.median(), color="green", label="Median")
    ax[0].axvline(rent, color="red", linewidth=2, label="Your rent")
    ax[0].set_title("Rents of similar houses")
    ax[0].legend()
    names = ["Your rent", "Your size", "Your rent per sq ft"]
    vals = [rent / mean * 100, size / d["Size"].mean() * 100, (rent / size) / ppsf.mean() * 100]
    ax[1].barh(names, vals, color=["#4338ca", "#16a34a", "#ea580c"])
    ax[1].axvline(100, linestyle="--", color="black")
    ax[1].set_title("Your readings vs average of similar houses (dashed = 100%)")
    plt.tight_layout()
    return c1, k, b, done(fig)


# ---------- Tab 2: Data Explorer ----------
def explore(cities, furns, bhk_min, bhk_max, rent_cap, chart):
    d = df[df["City"].isin(cities) & df["Furnishing_Status"].isin(furns) & df["BHK"].between(bhk_min, bhk_max) & (df["Rent"] <= rent_cap)]
    if len(d) < 3:
        return card("No data", "Nothing found", "Change the filters.", "#f87171"), None, pd.DataFrame(), None
    k = tiles([("Houses", format(len(d), ",")), ("Mean rent", inr(d["Rent"].mean())), ("Median rent", inr(d["Rent"].median())),
               ("Std deviation", inr(d["Rent"].std())), ("Skewness", format(d["Rent"].skew(), ".2f")),
               ("Average size", format(d["Size"].mean(), ".0f") + " sq ft")])
    fig, ax = plt.subplots(figsize=(9, 4))
    if chart == "Rent histogram":
        ax.hist(d["Rent"], bins=40, color="#6366f1", edgecolor="white")
        ax.axvline(d["Rent"].mean(), color="red", label="Mean")
        ax.axvline(d["Rent"].median(), color="green", label="Median")
        ax.legend()
        ax.set_xlabel("Rent")
    elif chart == "Log rent histogram":
        ax.hist(np.log(d["Rent"]), bins=40, color="#16a34a", edgecolor="white")
        ax.set_xlabel("log(Rent)")
    elif chart == "Average rent by city":
        d.groupby("City")["Rent"].mean().sort_values().plot(kind="barh", ax=ax, color="#4338ca")
    elif chart == "Average rent by furnishing":
        d.groupby("Furnishing_Status")["Rent"].mean().plot(kind="bar", ax=ax, color="#ea580c")
    elif chart == "Size vs Rent":
        s = d.sample(min(3000, len(d)), random_state=1)
        ax.scatter(s["Size"], s["Rent"], s=10, alpha=0.4, color="#4338ca")
        ax.set_xlabel("Size (sq ft)")
        ax.set_ylabel("Rent")
    else:
        d["Furnishing_Status"].value_counts().plot(kind="pie", ax=ax, autopct="%1.1f%%")
        ax.set_ylabel("")
    ax.set_title(chart)
    plt.tight_layout()
    table = d.sort_values("Rent", ascending=False).head(20)[["City", "BHK", "Size", "Furnishing_Status", "Bathroom", "Rent"]]
    path = "/tmp/filtered_rent_data.csv"
    d.to_csv(path, index=False)
    return k, done(fig), table, path


# ---------- Tab 3: Rent Category Predictor (uses the saved best classification model) ----------
saved = joblib.load("best_rent_model.pkl")
AREA_TYPES = sorted(df["Area_Type"].unique())
TENANTS = sorted(df["Tenant_Preferred"].unique())
CONTACTS = sorted(df["Point_of_Contact"].unique())
CAT_COLORS = {"Low": "#4ade80", "Medium": "#facc15", "High": "#f87171"}


def predict_category(city, furn, bhk, size, bath, floor_in, total_in, area, tenant, contact):
    row = pd.DataFrame([{"BHK": int(bhk), "Size": size, "Bathroom": int(bath), "Floor_No": floor_in,
                         "Total_Floors": total_in, "City": city, "Furnishing_Status": furn,
                         "Area_Type": area, "Tenant_Preferred": tenant, "Point_of_Contact": contact}])
    row = pd.get_dummies(row, columns=cat_cols).reindex(columns=saved["columns"], fill_value=0).astype(float)

    model = saved["model"]
    proba = model.predict_proba(row)[0]
    classes = list(model.classes_)
    pred = classes[int(np.argmax(proba))]

    edges = saved["rent_edges"]
    ranges = {"Low": (edges[0], edges[1]), "Medium": (edges[1], edges[2]), "High": (edges[2], edges[3])}
    lo, hi = ranges[pred]
    c1 = card("Predicted rent category", pred, "Typical monthly rent in this group: " + inr(lo) + " - " + inr(hi)
              + "<br>Model used: " + saved["model_name"], CAT_COLORS[pred])
    c2 = bars([(cl + " rent", p * 100, CAT_COLORS[cl]) for cl, p in zip(classes, proba)])
    return c1, c2

In [ ]:
#Dashboard part 2: layout (tabs) and public link
size_max = int(df["Size"].quantile(0.99))

with gr.Blocks(title="House Rent Analysis Dashboard") as demo:
    gr.Markdown("# 🏠 House Rent Analysis Dashboard\n"
                "Explore rent across Indian cities. "
                "**Note:** results are based on the Kaggle House Rent dataset (" + format(len(df), ",") + " listings).")

    with gr.Tabs():
        # ---- Tab 1
        with gr.Tab("🏠 Rent Checker"):
            gr.Markdown("Enter a house and its rent. We compare it with similar houses in the dataset.")
            with gr.Row():
                with gr.Column(scale=1):
                    c_city = gr.Dropdown(CITIES, value=CITIES[0], label="City")
                    c_furn = gr.Dropdown(FURN, value=FURN[0], label="Furnishing status")
                    c_bhk = gr.Dropdown(BHKS, value=BHKS[1] if len(BHKS) > 1 else BHKS[0], label="BHK")
                    c_size = gr.Slider(100, size_max, value=int(df["Size"].median()), step=10, label="Size (sq ft)")
                    c_rent = gr.Number(value=int(df["Rent"].median()), label="Your monthly rent (₹)")
                    c_conf = gr.Slider(80, 99, value=95, step=1, label="Confidence level (%)")
                    c_btn = gr.Button("Check my rent", variant="primary")
                with gr.Column(scale=2):
                    c_out1 = gr.HTML()
                    c_out2 = gr.HTML()
                    c_out3 = gr.HTML()
                    c_out4 = gr.Plot()
            c_in = [c_city, c_furn, c_bhk, c_size, c_rent, c_conf]
            c_out = [c_out1, c_out2, c_out3, c_out4]
            c_btn.click(rent_check, c_in, c_out)
            demo.load(rent_check, c_in, c_out)

        # ---- Tab 2
        with gr.Tab("📊 Data Explorer"):
            with gr.Row():
                with gr.Column(scale=1):
                    e_city = gr.CheckboxGroup(CITIES, value=CITIES, label="Cities")
                    e_furn = gr.CheckboxGroup(FURN, value=FURN, label="Furnishing status")
                    e_bmin = gr.Slider(min(BHKS), max(BHKS), value=min(BHKS), step=1, label="Min BHK")
                    e_bmax = gr.Slider(min(BHKS), max(BHKS), value=max(BHKS), step=1, label="Max BHK")
                    e_cap = gr.Slider(5000, RENT_CAP, value=RENT_CAP, step=5000, label="Maximum rent (₹)")
                    e_chart = gr.Dropdown(["Rent histogram", "Log rent histogram", "Average rent by city", "Average rent by furnishing",
                                           "Size vs Rent", "Furnishing share (pie)"], value="Rent histogram", label="Chart")
                    e_btn = gr.Button("Update", variant="primary")
                with gr.Column(scale=2):
                    e_out1 = gr.HTML()
                    e_out2 = gr.Plot()
                    e_out3 = gr.Dataframe(label="Top 20 costliest houses in this selection")
                    e_out4 = gr.File(label="Download filtered data (CSV)")
            e_in = [e_city, e_furn, e_bmin, e_bmax, e_cap, e_chart]
            e_out = [e_out1, e_out2, e_out3, e_out4]
            e_btn.click(explore, e_in, e_out)
            demo.load(explore, e_in, e_out)

        # ---- Tab 3
        with gr.Tab("🤖 Rent Category Predictor"):
            gr.Markdown("Enter the details of a house. The best classification model predicts whether its rent is **Low, Medium or High**.")
            with gr.Row():
                with gr.Column(scale=1):
                    p_city = gr.Dropdown(CITIES, value=CITIES[0], label="City")
                    p_furn = gr.Dropdown(FURN, value=FURN[0], label="Furnishing status")
                    p_bhk = gr.Dropdown(BHKS, value=BHKS[1] if len(BHKS) > 1 else BHKS[0], label="BHK")
                    p_size = gr.Slider(100, size_max, value=int(df["Size"].median()), step=10, label="Size (sq ft)")
                    p_bath = gr.Slider(1, int(df["Bathroom"].max()), value=2, step=1, label="Bathrooms")
                    p_floor = gr.Number(value=1, label="Floor number (0 = Ground)")
                    p_total = gr.Number(value=4, label="Total floors in building")
                    p_area = gr.Dropdown(AREA_TYPES, value=AREA_TYPES[0], label="Area type")
                    p_tenant = gr.Dropdown(TENANTS, value=TENANTS[0], label="Tenant preferred")
                    p_contact = gr.Dropdown(CONTACTS, value=CONTACTS[0], label="Point of contact")
                    p_btn = gr.Button("Predict rent category", variant="primary")
                with gr.Column(scale=2):
                    p_out1 = gr.HTML()
                    p_out2 = gr.HTML()
            p_in = [p_city, p_furn, p_bhk, p_size, p_bath, p_floor, p_total, p_area, p_tenant, p_contact]
            p_out = [p_out1, p_out2]
            p_btn.click(predict_category, p_in, p_out)


demo.launch(share=True, debug=True)

In [ ]:
#Conclusion

1. Rent is right skewed (Mean > Median); log of rent is close to normal.
2. Size, BHK and Bathroom have a positive correlation with Rent.
3. Sample means are close to the population mean, and the CLT holds for n = 30.
4. The 95% confidence interval for mean rent was calculated.
5. Z-test and T-test were used to test the claimed mean rent.
6. Chi-square and ANOVA show whether furnishing status and city affect the results.
7. Classification: three models (Logistic Regression, Decision Tree, Random Forest) were trained to predict the rent category (Low / Medium / High). They were compared using accuracy, precision, recall, F1-score and confusion matrix, and the best model was saved and used in the dashboard.
8. Regression: Linear Regression and Random Forest Regressor were trained to predict the exact monthly rent. They were compared using R², MAE and RMSE on the test set.